In [22]:
import pandas as pd

pd.set_option('display.max_rows', 10)

# Importing files

Two files are needed:
- the combined csv of all AllCalls data
- clarifications file (Clarifications_on_CAR_&_AllCallsData_6OCT.xlsx)

In the following code, manually input the path to these files on your computer.

In [23]:
# Read AllCalls dataset (with all cols as str dtype)
allcalls = pd.read_csv('Data/AllCallsData_combined.csv', dtype=str, na_filter=True)

In [24]:
# Read clarifications file, specifically the Popular numbers (AllCallsData) sheet
clarifications = pd.read_excel('Clarifications/Clarifications_on_CAR_&_AllCallsData_6OCT.xlsx', 
                               sheet_name='Popular numbers (AllCallsData)',
                               dtype={'Called number': str, 'Description': str})

# Cleaning AllCallsData

In [25]:
# Get rid of columns that are entirely NaN values
allcalls_no_nan = allcalls.dropna(axis=1, how='all')

# Update dtypes for columns that should not be represented as strings
allcalls_no_mixed = allcalls_no_nan.copy()

numeric_cols = ['Ring duration', 'Duration']
datetime_cols = ['Start time', 'Release time', 'Answer time', 'Report time']

for col in numeric_cols:
    if col in allcalls_no_mixed.columns:
        allcalls_no_mixed[col] = allcalls_no_mixed[col].astype(float)

for col in datetime_cols:
    if col in allcalls_no_mixed.columns:
        allcalls_no_mixed[col] = pd.to_datetime(allcalls_no_mixed[col], format='mixed')

**NOTE:** Although `dtype=str` is specified in the `read_csv` function to convert all the data into strings, `NaN` values are still left as floats. This means that some columns still technically have mixed data types.

# Processing for dashboard
This section involves creating supplemental things (e.g. new tables, new columns) that will be helpful for creating the Power BI dashboard.

## Editing the AllCalls table

In [26]:
# Select only cols relevant for visualizing the distribution of original/redirect reason for different called numbers
allcalls_reasons = allcalls_no_mixed[['Correlation ID', 'Called number', 'Redirecting number', 
                                      'Original reason', 'Related reason', 'Redirect reason', 
                                      'Inbound trunk', 'Outbound trunk', 'Direction', 
                                      'Call type', 'Client type', 'User type', 'Start time']].copy()

The following code block is to help with time-related filtering (e.g. if a call is during open/closed hours, if a call is on a weekday/weekend, etc.).

In [27]:
# Create version of 'Start time' localized to US/Central (DST is automatically accounted for)
allcalls_reasons['Start time (Chicago)'] = allcalls_reasons['Start time'].dt.tz_convert('America/Chicago')

# Create column indicating day of week
allcalls_reasons['Day of week'] = allcalls_reasons['Start time (Chicago)'].dt.day_name()

In [28]:
allcalls_reasons.head(3)

,Correlation ID,Called number,Redirecting number,Original reason,Related reason,Redirect reason,Inbound trunk,Outbound trunk,Direction,Call type,Client type,User type,Start time,Start time (Chicago),Day of week
0,91dada8b-2545-4a48-bf74-9e975e36859f,13123411070,NaN,NaN,NaN,NaN,NaN,wcc_Pc_tp-ipRwm_ku064NHZiw,TERMINATING,SIP_INBOUND,WXCC,Unknown,2024-04-30 23:58:53.988000+00:00,2024-04-30 18:58:53.988000-05:00,Tuesday
1,36826ea7-3d2f-4374-96e1-d208215b94ac,13123478300,NaN,NaN,NaN,NaN,NaN,NaN,TERMINATING,SIP_INBOUND,SIP,VoiceMailRetrieval,2024-04-30 23:56:37.386000+00:00,2024-04-30 18:56:37.386000-05:00,Tuesday
2,955117f1-8425-4c98-834b-24bc2e7885ed,13123478300,13123411070,NoAnswer,NaN,NoAnswer,NaN,NaN,TERMINATING,SIP_ENTERPRISE,SIP,VoiceMailRetrieval,2024-04-30 23:54:59.099000+00:00,2024-04-30 18:54:59.099000-05:00,Tuesday


## Phone number table

**Question:** Why are there phone numbers and descriptions placed randomly to the side in the clarifications file?

In [29]:
# Using the clarifications file, make a table linking phone numbers to descriptions
phone_nums_descriptions = clarifications.loc[clarifications['Description'].notna(), ['Called number', 'Description']]
phone_nums_descriptions.reset_index(drop=True, inplace=True)

# Include any called numbers that are in the top 10 most frequent numbers, even if there's no description
top_10_called_numbers = allcalls_reasons['Called number'].value_counts().head(10).index.to_list()
for num in top_10_called_numbers:
    if num not in phone_nums_descriptions['Called number'].to_list():
        phone_nums_descriptions.loc[len(phone_nums_descriptions)] = [num, '?']

# Add count column
phone_nums_val_counts = pd.DataFrame(allcalls_reasons['Called number'].value_counts()).reset_index()
phone_nums_descriptions = pd.merge(phone_nums_descriptions, phone_nums_val_counts, how='left').sort_values(by='count', ascending=False)

display(phone_nums_descriptions)

,Called number,Description,count
0,13123478300,Internal voicemail - not client related,235325
1,13123411070,Main number,222390
2,2302,Staff Directory English Transfer,56716
3,13124235938,legalclinics (Community Legal Clinics),56152
4,13122296300,Direct Line to Front Desk? Who is this given to?,36376
...,...,...,...
12,13123478340,Veterans Rights Project VM,2801
13,13122296014,Markham Eviction Help Desk,2386
14,13123478309,HIV Intake VM,1889
15,13122296072,JEHD (Juvenile Expungement Help Desk),1456


# Exporting

You can adjust where the file is exported to.

In [30]:
allcalls_reasons.to_csv('Data/AllCallsData_reasons.csv', index=False)
phone_nums_descriptions.to_csv('Data/PhoneNumberDescriptions.csv', index=False)